In [ ]:
import praw
import json
import csv
import re
import string
import datetime
import pandas as pd
import os
import time
import tqdm

In [ ]:
FILE_NAME = 'del.csv'

In [ ]:
path = os.path.join('../','data', FILE_NAME)

In [ ]:
with open('/Users/szymonleszkiewicz/Desktop/AI2024/AMC/client_secrets.json', 'r') as secrets_file:
    secrets = json.load(secrets_file)

reddit = praw.Reddit(
    client_id=secrets['client_id'],
    client_secret=secrets['client_secret'],
    refresh_token=secrets['refresh_token'],
    user_agent='MyRedditApp/1.0'
)

In [ ]:
def preprocess_text_simple(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^a-zżźćńółęąśŻŹĆĄŚĘŁÓŃ0-9\s]", "", text)
    text = re.sub(f"[{re.escape(string.punctuation)}]", '', text)
    return text

In [ ]:
csvfile = open(path, mode='w', newline='')
writer = csv.writer(csvfile, delimiter=',')
writer.writerow(['author', 'id', 'date', 'type', 'subreddit', 'content', 'ratio', 'score', 'replies'])

In [ ]:
def get_data(subreddit, hashtags, limit=None):
    counter = 0
    contributors = set()
    global writer

    # try to run the query else wait 30 seconds
    try:
        results  = subreddit.search(" OR ".join(hashtags), limit=limit)
    except:
        time.sleep(30)
        results  = subreddit.search(" OR ".join(hashtags), limit=limit)

    for submission in results:
        data = submission.created_utc
        data = datetime.datetime.fromtimestamp(data)
        if data < datetime.datetime(2023, 4, 1):
            continue
        
        if submission.author:
            contributors.add(submission.author)
            
        writer.writerow(
            [submission.author, submission.id, data, 'submission', submission.subreddit,
             preprocess_text_simple(submission.title),
             submission.upvote_ratio, submission.score, submission.num_comments])
        counter += 1 
        try:
            submission.comments.replace_more(limit=None)
        except:
            time.sleep(30)
            submission.comments.replace_more(limit=None)
        
        if counter > 2000 and subreddit.display_name == 'Polska':
            break
        com_counter = 0
        for comment in submission.comments.list():
            if comment.author:
                contributors.add(comment.author)
                counter += 1
                com_counter += 1
                writer.writerow([comment.author, submission.id, data, 'comment', comment.subreddit,
                                 preprocess_text_simple(comment.body), 1, comment.score, len(comment.replies)])
                if com_counter > 100 and subreddit.display_name == 'Polska':
                    break
                    
    return counter, contributors

In [ ]:
# read keywords
keywords = []
with open('/Users/szymonleszkiewicz/Desktop/AI2024/AMC/keywords.txt', 'r') as keywords_file:
    for line in keywords_file:
        keywords.append(line.strip())

In [ ]:
print('Keywords: ', len(keywords))

In [ ]:
subreddits = ['Polska', 'PolskaPolityka', 'libek', 'konfa', 'pis', 'The_Donek', 'poland', 'Polska_wpz', 'lewica']
limit = None

In [ ]:
for sub_name in subreddits:
    counter2 = 0
    contributors = set()
    print('Subreddit: ', sub_name)
    for i in tqdm.tqdm(range(0, len(keywords), 5)):
        users, con = get_data(reddit.subreddit(sub_name), keywords[i:i + 5], limit)
        contributors.update(con)
        counter2 += users
    print('records: ', counter2)
    print('contributors: ', len(contributors))

In [ ]:
csvfile.close()

In [ ]:
# read data 
df = pd.read_csv(path)
df.head()

In [ ]:
# get distinct users
df['author'].nunique()

In [ ]:
# get data range
df['date'].min(), df['date'].max()

In [ ]:
shape1 = df.shape

In [ ]:
# delete duplicates
df.drop_duplicates(inplace=True)
print(shape1, "-->", df.shape)

In [ ]:
# drop nulls
df.dropna(inplace=True)
print(shape1, "-->", df.shape)

In [ ]:
# save data
df.to_csv(path, index=False)

In [ ]:
# print unique subreddits
df['subreddit'].unique()

In [ ]:
# count records per subreddit
df['subreddit'].value_counts()

In [ ]:
# count authors per subreddit
df.groupby('subreddit')['author'].nunique()

In [ ]:
# show random 50 records with type = sumission and subreddit = maporn
df[(df['type'] == 'submission') & (df['subreddit'] == 'Polska')]['content'].sample(50)
